# 1. Package Imports section

In [0]:
import re
import logging
import pyspark.sql.functions as F
from pyspark.sql import Window

# 2. Dataset Configs

In [0]:
# logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_Service_Requests")

#silver tables config
ds_config = {
        "silver_table": "cpt_utility_catalog.silver.silver_service_requests_cleaned",
        "bronze_table": "cpt_utility_catalog.bronze.bronze_service_requests_raw",
        "changes": {
            "headers": {
                "column_mapping":{
                    "Sub_Council": "subcouncil",
                    "Ward": "ward",
                    "Suburb": "suburb",
                    "C3_Complaint_Type": "c3_complaint_type",
                    "Work_Center": "work_center",
                    "Notification": "notification",
                    "Notification_type": "notification_type",
                    "X_Y_Co_ordinate_1": "latitude_y",
                    "X_Y_Co_ordinate_2": "longitude_x",
                    "Created_On_Date": "created_on_date",
                    "Changed_on": "changed_on_date",
                    "Completed_Date": "completed_on_date",
                    "Notifications_Created": "notifications_created",
                }
            },
            "columns": {
                 "data_types":{
                    "subcouncil": "int",
                    "ward": "int",
                    "suburb": "string",
                    "c3_complaint_type": "string",
                    "notification": "int",
                    "notification_type": "string",
                    "longitude_x": "decimal(15,8)",
                    "latitude_y": "decimal(15,8)",
                    "created_on_date": "date",
                    "changed_on_date":"date",
                    "completed_on_date": "date",
                    "notifications_created":"int",
                    "work_center": "string"
                },
                "columns_to_drop": ["ObjectId"],
                "fill_na_value": None, 
            },
            "trim": True,
            "drop_columns": True,
            "drop_duplicates": True,
            "write_to_table": True,
            "rename_headers": True,
            "unpivot": True,
            "add_id": True,
            "cast_data_types": True,
            "col_cleanse": True,
        },
    }

df_new = spark.table(ds_config["bronze_table"])
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]
changes = ds_config["changes"]

logger.info("\t- Silver layer service requests table configuration loaded")


# 3. Dataset Cleaning

## 3.1 Drop Columns

In [0]:
if changes["drop_columns"] and isinstance(col_config["columns_to_drop"], list):
    logger.info("Dropping column(s)")
    # dropping columns that won't be needed acccording to config
    df_new = df_new.drop(*col_config["columns_to_drop"])
    logger.info("\t- Column(s) dropped")

## 3.2 Rename Headers

In [0]:
if changes["rename_headers"]:
    logger.info("Renaming headers")
    hdr_transformations = dict()
    # enrishing dataset with sensible names
    if isinstance(hdr_config["column_mapping"], dict):

        df_new = df_new.select(
        [F.col(col).alias(hdr_config["column_mapping"].get(col, col)) for col in df_new.columns]
    )

    logger.info("\t- Header(s) renamed")

## 3.3 Column Cleanse

In [0]:
if changes["col_cleanse"]:
        logger.info("Cleaning column(s)")

        df_new = df_new.withColumn(
            "subcouncil",
            F.regexp_replace(
                F.col("subcouncil"),
                r"Sub-Council\s+",
                ""
            )
        )

        # Turn exact "000" matches into NULL using regex
        df_new = df_new.withColumn(
        "ward",
        F.when(F.col("ward").rlike(r"^000$"), F.lit(None))
        .otherwise(F.col("ward"))
        )

        # Turn exact "0" matches into NULL using regex
        df_new = df_new.withColumn(
        "subcouncil",
        F.when(F.col("subcouncil").rlike(r"^0$"), F.lit(None))
        .otherwise(F.col("subcouncil"))
        )

        # Turn exact "0" matches into NULL using regex
        df_new = df_new.withColumn(
        "longitude_x",
        F.when(F.col("longitude_x").rlike(r"^0$"), F.lit(None))
        .otherwise(F.col("longitude_x"))
        )

        # Turn exact "0" matches into NULL using regex
        df_new = df_new.withColumn(
        "latitude_y",
        F.when(F.col("latitude_y").rlike(r"^0$"), F.lit(None))
        .otherwise(F.col("latitude_y"))
        )

        # Turn exact "0" matches into NULL using regex
        df_new = df_new.withColumn(
                "suburb",
                F.upper(F.col("suburb"))
        )

        # Turn exact digits matches into NULL using regex
        df_new = df_new.withColumn(
        "notification_type",
        F.when(F.col("notification_type").rlike(r"^\d+$"), F.lit(None))
        .otherwise(F.col("notification_type"))
        )
        
        # Replace any occurrence of these placeholders with SQL NULL
        bad_placeholders = ["#", "#_Not assigned", "Not assigned","NUL", "NULL", "N/A", "NONE", ""]

        df_new = df_new.replace(bad_placeholders, None)

        # Insert a space between date (yyyy/MM/dd) without time (HH:mm:ss) since all is 22:00:00
        df_new = df_new.withColumn(
            "created_on_date",
            F.regexp_replace(
                F.col("created_on_date"), 
                r"\s+\d{2}:\d{2}:\d{2}[+-].*", 
                ""
            )
        )
        
        df_new = df_new.withColumn(
            "changed_on_date",
            F.regexp_replace(
                F.col("changed_on_date"), 
                r"\s+\d{2}:\d{2}:\d{2}[+-].*", 
                ""
            )
        )

        df_new = df_new.withColumn(
            "completed_on_date",
            F.regexp_replace(
                F.col("completed_on_date"), 
                r"\s+\d{2}:\d{2}:\d{2}[+-].*", 
                ""
            )
        )

        philippi_pattern = r"P+[Hh]?I[Ll]{1,2}[Ii]*[Pp]{1,3}[A-Za-z]*"

        df_new = df_new.withColumn(
            "suburb",
            F.when(F.col("suburb").rlike(philippi_pattern), "PHILIPPI")
            .otherwise(F.col("suburb"))
        )

        khayelitsha_pattern = r"K+[Hh]?[AaEeIu]*[Yy][AaEe]*L+[EeIi]*[TtCcDdSs]*[SsHh]+[AaEe]*"

        df_new = df_new.withColumn(
            "suburb",
            F.when(F.col("suburb").rlike(khayelitsha_pattern), "KHAYELITSHA")
            .otherwise(F.col("suburb"))
        )

        gugulethu_pattern = r"1?G+[Uu]+G+[A-Za-z]*"

        df_new = df_new.withColumn(
            "suburb",
            F.when(F.col("suburb").rlike(gugulethu_pattern), "GUGULETHU")
            .otherwise(F.col("suburb"))
        )
        
        # stadardising suburb names and cleaning errors

        bad_placeholders = ["#_Not assigned","0","Not assigned","NUL", "NULL", "N/A", "NONE","GENERIC TOWN 0000","UNKNOWN"]

        df_new = df_new.replace(bad_placeholders, None, subset=["suburb"])

        correct_suburb = {
            "PHILIPI": "PHILIPPI",
            "SITE C": "KHAYELITSHA",
            "CAPE TOWN": "CAPE TOWN CITY CENTRE",
            "PARKWOOD EST": "PARKWOOD",
            "BELLVILLE-SUID": "BELLVILLE SOUTH",
            "CAMPS BAY": "CAMPS BAY / BAKOVEN",
            "WESTRIDGE - MITCHELLS PLA": "WESTRIDGE - MITCHELLS PLAIN",
            "RUSTHOF": "STRAND",
            "SCOTTSDENE, KRAAIFONTEIN": "KRAAIFONTEIN",
            "KLEINVLEI": "KLEINVLEI TOWN",
            "LOCHNERHOF": "STRAND",
            "BROWNS FARM": "PHILIPI",
            "PARKLANDS EXT": "PARKLANDS",
            "SYBRANDPARK": "SYBRAND PARK",
            "SIR LOWRYS PASS": "SIR LOWRY'S PASS",
            "EERSTE RIVER": "EERSTERIVIER",
            "NYANGA EAST": "NYANGA",
            "BELLVILLE SOUTH EXT 13": "BELLVILLE SOUTH",
            "SURREY": "SURREY ESTATE",
            "HOUTBAY": "HOUT BAY",
            "SCOTTSDENE KRAAIFONTEIN": "KRAAIFONTEIN",
            "BAKOVEN": "CAMPS BAY / BAKOVEN",
            "PINATI": "PINATI ESTATE",
            "MELKBOSCH STRAND": "MELKBOSSTRAND",
        }

        df_new = df_new.replace(correct_suburb, subset=["suburb"])



        
        logger.info("\t- Column(s) cleaned")

## 3.4 Trim Whitespace

In [0]:
if changes["trim"]:
    logger.info("Trimming column(s)")
    # trimming column values whilst they're all casted string
    df_new = df_new.select([F.trim(F.col(col)).alias(col) for col in df_new.columns])
    logger.info("\t- Column(s) trimmed")

## 3.5 Cast Data Types

In [0]:
if changes["cast_data_types"]:
    logger.info("Casting Datatype(s)")
    # casting data types according to the config
    df_new = df_new.select([
            F.coalesce(
                F.try_to_date(F.col(col), "yyyy/MM/dd"),
                F.try_to_date(F.col(col), "dd/MM/yyyy"),
            ).alias(col) if col in ["created_on_date", "changed_on_date", "completed_on_date"]
            else F.col(col).try_cast(col_config["data_types"][col]).alias(col) 
            for col in df_new.columns
           
     ])
    logger.info("\t- Datatype(s) casted")

## 3.6 Adding Primary key

In [0]:
if changes["add_id"]:
    logger.info("Adding ID(s)")
    # creating primary key column with md5 hash of main columns   
    df_new = df_new.withColumn(
        "id",
        F.md5(
            F.concat_ws(
                "|",
                F.col("suburb"),
                F.col("notification"),
                F.col("c3_complaint_type"),
                F.col("notification_type"),
                F.col("created_on_date"),
                F.col("completed_on_date")
            )
        ),
    )
    logger.info("\t- ID(s) added")

## 3.7 Drop Duplicates

In [0]:
if changes["drop_duplicates"]:
    logger.info("Dropping Duplicate(s)")

    # Ensure empty strings in integer/text columns are NULL before window operations
    df_new = df_new.replace("", None)

    # Exclude key/date columns from the NULL count check
    exclude_cols = ["id", "notification", "created_on_date", "changed_on_date", "completed_on_date"]
    cols_to_check = [c for c in df_new.columns if c not in exclude_cols]

    # Count NULLs across payload columns
    null_count_expr = sum(
        [F.when(F.col(c).isNull(), 1).otherwise(0) for c in cols_to_check]
    )

    df_with_nulls = df_new.withColumn("null_count", null_count_expr)

    # Partition by unique ticket key ('id' or 'notification') so distinct tickets aren't deleted
    window_spec = Window.partitionBy("id").orderBy(F.col("null_count").asc())

    # Keep the row with the lowest null count per unique ID
    df_deduped = (
        df_with_nulls.withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num", "null_count")
    )

    df_new = df_deduped
    logger.info("\t - Duplicate(s) Dropped")
    print(df_new.count())

# 4. Writing  To Silver Layer

In [0]:
if changes["write_to_table"]:    
    logger.info("Writing service requests to Silver Table")
    target_table_name = ds_config["silver_table"]

    # 1. Register your cleaned DataFrame as a temporary view for SQL execution
    df_new.createOrReplaceTempView("temp_source_data")

    if spark.catalog.tableExists(target_table_name):
        # 2. Run the native Databricks SQL Merge command
        spark.sql(f"""
            MERGE INTO {target_table_name} AS target
            USING temp_source_data AS source
            ON target.id = source.id
            WHEN MATCHED THEN 
                UPDATE SET *
            WHEN NOT MATCHED THEN 
                INSERT *
        """)
        logger.info("\t - Delta table successfully upserted.")
    else:
        # First-time run: Create the table
        (
            df_new.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table_name)
        )
        logger.info("\t - Target table did not exist. Created new Delta table.")
    logger.info("\t - suburb service requests to Silver Table")